# 강의 06 · 실습 7 — 패턴 1 병렬화 · (1) 강사 시연

## 1. 문제상황

- 총무팀은 사내 공지를 낼 때마다 공지 초안을 대강 적어 두고, 제목과 핵심 요약과 직원 준비물을 따로 정리합니다.
- 담당자는 에이전트 하나에 초안을 주고 제목·요약·준비물을 한꺼번에 만들어 달라고 시켰습니다.
- 같은 초안을 넣어도 물을 때마다 세 부분의 내용과 형식이 달라지고, 세 부분을 차례로 만드느라 답이 늦게 옵니다.
- 공지가 늘어나면 담당자는 매번 결과를 다시 손보는 일을 그만큼 반복해야 합니다.

## 2. 문제와 목표

- **문제**: 성격이 다른 세 가지 작성 작업(제목·요약·준비물)을 에이전트 하나가 한 번에 맡습니다. 결과가 매번 달라지고, 세 작업이 서로를 기다리므로 느립니다.
- **목표**: 작성 역할을 셋으로 나누고, 공지 초안만 넣으면 세 노드가 동시에 각자의 결과를 만들고, 결합 노드가 정해진 순서로 이어 붙여 공지문을 완성하는 처리 흐름을 만듭니다.
    - 세 노드: summary(핵심 요약 한 문장), checklist(직원이 미리 할 일 2개), title(20자 이내 제목).
    - 결합 노드: aggregator — 모델을 부르지 않고 제목, 요약, 빈 줄, 「준비할 것:」, 체크리스트 순서로 이어 붙입니다.
- **목표 달성 여부의 판정 기준**: 공지 초안을 입력했을 때, 세 작성 노드가 서로를 기다리지 않고 같은 단계에서 진입하고, 결합 노드가 셋이 끝난 뒤에 한 번만 도는 것을 실행 결과에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec06_ex07_s1_diagram.svg)

## 4. 단계별 요구사항

1. **상태를 정의합니다.**
    - 공지 초안(`draft`), 핵심 요약(`summary`), 준비물 체크리스트(`checklist`), 제목 후보(`title`), 합쳐진 공지문(`notice`) 키 다섯 개를 가지는 상태를 선언합니다.
    - 키 다섯 개 외의 값은 상태에 들어가지 않습니다.
2. **작성 노드 세 개를 만듭니다.**
    - summary 노드는 초안의 핵심을 한 문장으로 요약해 `summary` 키에 씁니다.
    - checklist 노드는 직원이 미리 할 일 2개를 뽑아 `checklist` 키에 씁니다.
    - title 노드는 20자 이내 제목을 지어 `title` 키에 씁니다.
    - 세 노드는 같은 초안을 입력으로 받되 시스템 프롬프트가 다르고, 서로 다른 키에만 씁니다.
3. **결합 노드를 만듭니다.**
    - aggregator 노드는 모델을 부르지 않고, 상태의 제목·요약·체크리스트를 정해진 순서(제목, 요약, 빈 줄, 준비할 것, 체크리스트)로 이어 붙여 `notice` 키에 씁니다.
4. **그래프에 노드를 등록합니다.**
    - 네 노드를 이름과 함께 그래프에 등록합니다.
5. **엣지를 연결합니다.**
    - START에서 summary·checklist·title 세 노드로 가는 고정 엣지를 각각 놓습니다.
    - 세 노드에서 aggregator로 가는 고정 엣지를 각각 놓습니다.
    - aggregator 뒤에는 END를 고정 엣지로 연결합니다.
    - 조건부 엣지는 추가하지 않습니다.
6. **그래프를 컴파일하고 실행합니다.**
    - 공지 초안을 넣고, 노드가 하나 끝날 때마다 어느 노드가 상태의 어느 키를 채웠는지 화면에 출력한 뒤, 완성된 공지문을 출력합니다.

## 5. 코드 골격 — LangGraph 5단

랭그래프(LangGraph)로 그래프를 세우는 순서는 다음 다섯 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 다섯 단계와 하나씩 대응합니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 상태 정의 | 노드들이 함께 읽고 쓸 키를 선언합니다 | `class NoticeState(TypedDict)` | 1 |
| ② 노드 함수 정의 | 상태를 받아 바뀐 키만 돌려주는 함수를 만듭니다 | `def write_summary(state) -> dict` | 2, 3 |
| ③ 그래프 빌더 생성과 노드 등록 | 빈 그래프를 열고 함수에 이름을 붙여 등록합니다 | `StateGraph(NoticeState)`, `add_node` | 4 |
| ④ 엣지 연결 | START에서 나뉘고 한 곳으로 모이는 순서를 정합니다 | `add_edge` | 5 |
| ⑤ 컴파일과 실행 | 연결을 확정하고 입력을 넣어 실행합니다 | `compile()`, `stream()` | 6 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv
from typing import TypedDict

from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
print("모델 준비를 마쳤습니다.")

### 단계 ① — 상태 정의 (요구사항 1)

그래프가 도는 동안 모든 노드가 함께 읽고 쓰는 키를 선언합니다. `TypedDict`로 선언한 다섯 개의 키가 이 그래프에서 오가는 데이터의 전부입니다. 세 작성 노드가 각각 다른 키에 쓰도록 `summary`·`checklist`·`title` 키 세 개를 따로 둡니다.

In [ ]:
class NoticeState(TypedDict):
    draft: str      # 공지 초안(내용을 대강 적은 메모)
    summary: str    # 핵심 요약
    checklist: str  # 준비물 체크리스트
    title: str      # 제목 후보
    notice: str     # 합쳐진 공지문


print("상태의 키:", list(NoticeState.__annotations__))

### 단계 ② — 노드 함수 정의 (요구사항 2, 3)

- 노드는 상태를 인자로 받아 딕셔너리를 돌려주는 파이썬 함수입니다. 돌려준 딕셔너리가 상태의 해당 키를 덮습니다.
- 세 작성 노드는 같은 초안을 `HumanMessage`로 받지만, `SystemMessage`가 서로 다르므로 서로 다른 결과를 만듭니다. 시스템 프롬프트가 다른가가 노드를 나누는 기준입니다.
- 세 작성 노드는 서로의 결과를 읽지 않습니다. 그래서 동시에 돌 수 있습니다.
- aggregator 노드는 모델을 부르지 않습니다. 상태에 담긴 세 결과를 문자열로 이어 붙이기만 합니다.

In [ ]:
def write_summary(state: NoticeState) -> dict:
    """공지 초안의 핵심을 한 문장으로 요약한다."""
    print("  [summary] 진입")
    res = llm.invoke([
        SystemMessage("공지 초안의 핵심을 한 문장으로 요약한다."),
        HumanMessage(state["draft"]),
    ])
    return {"summary": res.content.strip()}


def write_checklist(state: NoticeState) -> dict:
    """직원이 미리 할 일 2개를 뽑는다."""
    print("  [checklist] 진입")
    res = llm.invoke([
        SystemMessage("공지 초안에서 직원이 미리 할 일 2개를 짧게 뽑는다."),
        HumanMessage(state["draft"]),
    ])
    return {"checklist": res.content.strip()}


def write_title(state: NoticeState) -> dict:
    """20자 이내 제목을 짓는다."""
    print("  [title] 진입")
    res = llm.invoke([
        SystemMessage("공지 초안에 어울리는 제목을 20자 이내 한 줄로 쓴다."),
        HumanMessage(state["draft"]),
    ])
    return {"title": res.content.strip()}


def aggregator(state: NoticeState) -> dict:
    """세 결과를 정해진 순서로 이어 붙인다 (모델을 부르지 않는다)."""
    print("  [aggregator] 진입 (셋이 모두 끝난 뒤)")
    notice = (f"[{state['title']}]\n{state['summary']}\n\n"
              f"준비할 것:\n{state['checklist']}")
    return {"notice": notice}

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 4)

`StateGraph`에 상태를 넘겨 빈 그래프를 열고, `add_node`로 함수마다 이름을 붙여 등록합니다. 여기서 붙인 이름은 뒤의 엣지 연결에서 그대로 쓰입니다.

In [ ]:
g = StateGraph(NoticeState)
g.add_node("summary", write_summary)
g.add_node("checklist", write_checklist)
g.add_node("title", write_title)
g.add_node("aggregator", aggregator)

print("등록한 노드:", list(g.nodes))

### 단계 ④ — 엣지 연결 (요구사항 5)

`add_edge`는 고정된 순서로 연결합니다. START에서 세 노드로 가는 엣지가 세 개이므로 세 노드는 같은 단계에서 동시에 시작합니다. 세 노드에서 aggregator로 가는 엣지가 세 개이므로 aggregator는 셋이 모두 끝난 뒤에 한 번 돕니다. 병렬 경로가 세 개라는 사실이 엣지 연결 코드에 그대로 적혀 있습니다.

In [ ]:
for name in ("summary", "checklist", "title"):
    g.add_edge(START, name)          # START에서 셋으로 나뉜다
    g.add_edge(name, "aggregator")   # 세 분기가 한 곳으로 모인다
g.add_edge("aggregator", END)

print("연결한 엣지 수:", len(g.edges))

### 단계 ⑤ — 컴파일과 실행 (요구사항 6)

`compile()`이 연결을 확정해 실행 가능한 그래프를 돌려줍니다. `stream`은 노드가 하나 끝날 때마다 그 노드가 바꾼 키를 내보냅니다. 아래에서는 자율좌석제 시행 공지의 초안을 넣습니다.

In [ ]:
graph = g.compile()

DRAFT = ("10월 6일부터 3·4층 자율좌석제 시행. "
         "예약은 포털 '좌석' 메뉴, 전일 17시부터. "
         "개인 물품은 사물함 보관. 문의 내선 0000.")

print(f"=== 입력 초안: {DRAFT[:30]}... ===")
final = {"draft": DRAFT}
for step in graph.stream({"draft": DRAFT}, stream_mode="updates"):
    for node, patch in step.items():
        print(f"  [{node}] -> 채운 키: {list(patch)}")
        final.update(patch)

print()
print("[최종 상태의 키]", sorted(final))
print("--- 공지문 ---")
print(final["notice"])

## 7. 실행 결과 확인

위 실행 결과에서 다음 세 가지를 확인합니다.

1. `summary`·`checklist`·`title` 세 노드의 「진입」 줄이 `aggregator`의 진입 줄보다 먼저 출력됩니다. 세 노드끼리의 순서는 실행할 때마다 달라질 수 있습니다. 서로를 기다리지 않기 때문입니다.
2. `aggregator`의 진입 줄은 마지막에 한 번만 출력됩니다. 세 노드가 모두 끝난 뒤에 도는 팬인(fan-in) 지점입니다.
3. 최종 상태의 키는 `checklist`·`draft`·`notice`·`summary`·`title` 다섯 개이고, 공지문은 제목, 요약, 빈 줄, 「준비할 것:」, 체크리스트의 순서로 이어져 있습니다. 순서는 모델이 아니라 aggregator 함수가 정했습니다.

세 작성 노드의 「채운 키」가 서로 다른 것이 병렬화가 멈추지 않고 동작한 증거입니다.